# Zeros na demanda e escolha da granularidade

Este notebook responde duas perguntas:

- **Parte A — Por que a demanda tem tantos zeros?** Se fosse erro de dado, seria preciso corrigir antes de modelar.
- **Parte B — Em que granularidade prever?** Diária, semanal, mensal ou trimestral, e qual valor exato prever.

**Resposta curta:** os zeros são venda baixa de verdade, não erro. A previsão será feita na **série mensal por produto × loja**, prevendo o **total dos 3 meses seguintes**.

**Em palavras simples:** cada produto, em cada loja, vende muito pouco — em geral só alguns dias por ano. Por isso, se você olhar dia a dia, quase todo dia vai ter "zero vendido" para aquele produto naquela loja. Isso não é um problema nos dados; é assim mesmo que esse tipo de negócio vende (poucos itens, com pedidos espaçados). A pergunta prática deste notebook é: em que "janela de tempo" (dia? semana? mês? trimestre?) e prevendo o quê (um mês? três meses?) fica mais fácil enxergar um padrão em vez de só ver zeros?

**Como ler:** cada seção traz a pergunta, como foi medido, a célula de cálculo e um **Resultado** com o que os números mostram. Os resultados foram observados em `dev_lh_nautical` na data indicada; se os dados mudarem, confira contra as saídas das células.

| Termo | Significado |
|---|---|
| Série | A demanda de um produto numa loja ao longo do tempo (6.054 séries) |
| Demanda | Quantidade vendida em pedidos confirmados ou pagos |
| Grão | O tamanho de cada período da série: dia, semana, mês ou trimestre |
| Taxa de zeros | Fração dos períodos sem nenhuma venda |
| Período incompleto | O primeiro período de cada série (começa no dia da primeira venda) e o último do dataset (pode estar pela metade) |

As quatro tabelas (`agg_demand_daily`, `_weekly`, `_monthly`, `_quarterly`, schema `ml`) vêm da mesma lógica: a diária registra zero em todo dia sem venda, desde a primeira venda da série até o último dia do dataset, e as outras só somam a diária em períodos maiores. Qualquer diferença entre elas é efeito do grão.

Os cálculos ficam em `src/analytics_ai/eda_demand_zeros.py`.

## Setup

In [ ]:
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Sobe a partir do diretório do notebook até achar a raiz do bundle (databricks.yml)."""
    for candidate in (start, *start.parents):
        if (candidate / "databricks.yml").exists():
            return candidate
    raise FileNotFoundError(f"databricks.yml não encontrado acima de {start}")


# Funciona no workspace (.bundle/.../files/notebooks/ml) e localmente.
PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import pandas as pd
from pyspark.sql import functions as F

from src.analytics_ai.eda_demand_zeros import (
    add_partial_period_flag,
    add_segment_columns,
    aggregate_series,
    demand_pattern_classification,
    grain_scorecard,
    horizon_target_comparison,
    order_status_filter_impact,
    seasonal_signal_share,
    summarize_demand_patterns,
    zero_rate_by,
    zero_rate_by_volume_decile,
    zero_rate_overall,
    zero_rows_from_trailing_dead_combinations,
)
from src.analytics_ai.ml_config import MlFeatureConfig

# No Databricks o notebook já expõe `spark`; localmente cria via Databricks Connect
# (defina DATABRICKS_PROFILE com o nome do seu profile em ~/.databrickscfg).
spark = globals().get("spark")
if spark is None:
    from src.ingestion.utils import build_spark_session

    spark = build_spark_session()

# Databricks e Jupyter injetam `display`; fora deles, cai para print.
display = globals().get("display", print)

# Grãos do mais fino ao mais agregado.
GRAINS = MlFeatureConfig.all_grains()
DFS = {name: spark.table(config.source_table) for name, config in GRAINS.items()}
daily_config, daily_df = GRAINS["diario"], DFS["diario"]

for name, config in GRAINS.items():
    print(f"{name:<11} {config.source_table}")

# Parte A — Por que tantos zeros?

## A1. Quanto da demanda é zero em cada grão?

**Como medimos:** fração dos períodos sem venda em cada uma das quatro tabelas.

In [ ]:
pd.concat(
    [
        zero_rate_overall(df, GRAINS[name].target_col, GRAINS[name].id_cols).toPandas().assign(grao=name)
        for name, df in DFS.items()
    ]
).set_index("grao")

**Resultado** *(16/09/2026)*

| Grão | Períodos sem venda |
|---|---|
| Diário | 99,2% |
| Semanal | 94,2% |
| Mensal | 77,8% |
| Trimestral | 48,5% |

Agregar reduz bastante os zeros, mas mesmo no mensal 3 em cada 4 meses de uma série não têm venda. As seções seguintes investigam a causa.

## A2. Os zeros se concentram em algum grupo?

**Como medimos:** taxa de zeros no grão diário, separada por alta temporada, categoria e tipo de loja. O diário é usado porque cada dia fica inteiro dentro ou fora da alta temporada, enquanto uma semana ou um mês pode ficar dividido.

**Por que importa:** se um grupo concentrasse os zeros, haveria uma causa específica para investigar ou tratar à parte.

In [ ]:
daily_segmented = add_segment_columns(daily_df, daily_config.gold_table, date_col=daily_config.date_col)

zero_rate_by(daily_segmented, ["is_high_season"]).show(truncate=False)
zero_rate_by(daily_segmented, ["location_type"]).show(truncate=False)
zero_rate_by(daily_segmented, ["category_name"]).show(50, truncate=False)

**Resultado** *(16/09/2026)*

- **Alta temporada:** 98,8% de dias sem venda, contra 99,5% fora dela. A diferença parece pequena, mas não significa que a estação é fraca: com chance de venda tão baixa por dia, até uma demanda muito maior aparece como diferença pequena na taxa de zeros. A B4 mostra que a estação é forte.
- **Tipo de loja:** loja e armazém dividem os zeros meio a meio (99,15% nos dois).
- **Categoria:** todas ficam entre 99,1% e 99,2%.

**O que significa:** nenhum grupo concentra os zeros. Não há um segmento problemático para tratar à parte.

## A3. Produtos que vendem mais têm menos zeros?

**Como medimos:** as séries são ordenadas pelo volume total e divididas em 10 faixas iguais (decis). Decil 1 = as 10% que menos vendem; decil 10 = as 10% que mais vendem. A tabela mostra a taxa de zeros de cada decil em cada grão.

**Por que importa:** se os zeros fossem só dos produtos de venda baixa, bastaria separar esses produtos.

In [ ]:
deciles = pd.concat(
    [
        zero_rate_by_volume_decile(df, id_cols=GRAINS[name].id_cols)
        .select("volume_decile", "zero_rate")
        .toPandas()
        .assign(grao=name)
        for name, df in DFS.items()
    ]
)

deciles.pivot(index="volume_decile", columns="grao", values="zero_rate")[list(GRAINS)]

**Resultado** *(16/09/2026)*

| Grão | 10% que menos vendem | 10% que mais vendem |
|---|---|---|
| Diário | 99,4% | 98,9% |
| Semanal | 96,0% | 92,4% |
| Mensal | 84,1% | 71,7% |
| Trimestral | 60,5% | 38,5% |

**O que significa:** até os produtos que mais vendem passam quase todos os dias sem venda. A diferença entre alto e baixo giro só aparece a partir do mensal. Os zeros não são exclusividade de um grupo de produtos fracos: é o jeito como esse negócio vende.

## A4. Os zeros vêm de produtos que pararam de vender?

**Como medimos:** uma série é considerada **parada** quando a última venda aconteceu mais de 90 dias antes do fim do dataset. A tabela mostra quantas séries estão paradas e qual fração dos zeros vem dos períodos depois da última venda delas.

**Por que importa:** a tabela diária registra zero até o fim do dataset mesmo para produtos que saíram de linha. Se essa fração fosse grande, a correção seria no dbt, não no modelo.

In [ ]:
pd.concat(
    [
        zero_rows_from_trailing_dead_combinations(df, date_col=GRAINS[name].date_col, dead_after_days=90)
        .toPandas()
        .assign(grao=name)
        for name, df in DFS.items()
    ]
).set_index("grao")[["n_combinations", "n_dead_combinations", "trailing_dead_share_of_zeros"]]

**Resultado** *(16/09/2026)*

- 3.519 das 6.054 séries (58%) estão sem venda há mais de 90 dias.
- Mesmo assim, os períodos depois da última venda delas respondem por só **4,7% dos zeros** no diário (5,0% no semanal, 6,1% no mensal, 8,1% no trimestral).

**O que significa:** produtos parados não explicam os zeros. Corrigir a tabela diária para parar de registrar zero após a última venda seria só uma limpeza, sem mudar o quadro. Para o modelo, séries paradas ficam fora do treino e recebem previsão zero por regra.

## A5. Algum grão deixa as séries regulares?

**Como medimos:** cada série é classificada pelo intervalo médio entre períodos com venda e pela variação da quantidade quando vende (classificação de Syntetos-Boylan):

| Classe | Significado |
|---|---|
| smooth | Vende quase todo período, em quantidade estável — o caso fácil de prever |
| intermittent | Muitos períodos sem venda, quantidade estável quando vende |
| erratic | Vende quase todo período, mas em quantidade muito variável |
| lumpy | Muitos períodos sem venda e quantidade muito variável — o caso mais difícil |

A primeira tabela mostra a fração de séries em cada classe; a segunda, o intervalo médio entre vendas, em períodos do próprio grão.

In [ ]:
patterns = pd.concat(
    [
        summarize_demand_patterns(demand_pattern_classification(df, date_col=GRAINS[name].date_col))
        .toPandas()
        .assign(grao=name)
        for name, df in DFS.items()
    ]
)

display(patterns.pivot(index="demand_pattern", columns="grao", values="share_of_combinations")[list(GRAINS)].fillna(0))
patterns.pivot(index="demand_pattern", columns="grao", values="avg_adi")[list(GRAINS)].round(1)

**Resultado** *(16/09/2026)*

| Grão | intermittent | lumpy | smooth | Intervalo médio entre vendas |
|---|---|---|---|---|
| Diário | 98,2% | 1,8% | 0% | 123 dias |
| Semanal | 95,7% | 4,3% | 0% | 18 semanas |
| Mensal | 86,4% | 13,6% | 0% | 4,7 meses |
| Trimestral | 71,2% | 28,3% | 0,4% | 2,0 trimestres |

**O que significa:** em nenhum grão as séries ficam regulares — praticamente não há série `smooth`. Agregar encurta o intervalo entre vendas, mas também transforma séries intermitentes em irregulares (lumpy): quando um período junta vários pedidos, a quantidade passa a variar muito. A série de um produto numa loja é **intermitente em qualquer grão**, e o modelo precisa lidar com isso.

**Em palavras simples:** pense em quatro tipos de torneira: **smooth** é a que pinga o tempo todo, sempre na mesma velocidade (fácil de prever) — não existe nenhuma assim aqui. **Intermittent** é a que fica seca boa parte do tempo, mas quando pinga, pinga sempre a mesma quantidade — é a maioria das nossas séries. **Erratic** pinga sempre, mas às vezes um fiozinho, às vezes um jato. **Lumpy** é a pior combinação: fica seca a maior parte do tempo, e quando pinga, a quantidade varia muito — é o que quase 30% das séries viram no grão trimestral. Junte pedidos de 3 meses numa linha só, e você mistura um mês de venda alta com dois secos, criando essa variação. **Conclusão prática:** não existe um grão "mágico" que deixe a venda de um produto isolado previsível e estável — isso é característica do negócio (poucas vendas, espaçadas), não um defeito a corrigir.

## A6. A definição de demanda está descartando vendas?

**Como medimos:** a demanda só conta itens de pedidos confirmados ou pagos. A tabela mostra quanto da quantidade cada situação de pedido representa e se entra na demanda (`included_in_demand`).

**Por que importa:** se o filtro descartasse muita venda real, parte dos zeros seria criada pela própria regra.

In [ ]:
order_status_filter_impact(spark.table(daily_config.gold_table("fact_order_items"))).show(truncate=False)

**Resultado** *(16/09/2026)*

| Situação do pedido | Entra na demanda | Fatia da quantidade |
|---|---|---|
| Pago | Sim | 70,1% |
| Confirmado | Sim | 15,1% |
| Cancelado | Não | 9,8% |
| Rascunho | Não | 4,9% |

**O que significa:** a regra descarta ~15% da quantidade, formada por pedidos cancelados e rascunhos. É uma exclusão correta e não explica 99% de dias sem venda.

## Resposta da Parte A

**Os zeros são venda baixa de verdade, não erro de dado.**

- Não vêm de um grupo específico (A2), nem só dos produtos fracos (A3), nem de produtos parados (A4), nem da regra de demanda (A6).
- A causa é o volume: uma série típica tem ~19 dias com venda em ~6 anos.
- A série de um produto numa loja é intermitente em qualquer grão (A5).

**Consequência:** os zeros ficam nos dados. O trabalho é escolher uma forma de prever que funcione com eles (Parte B).

# Parte B — Em que granularidade prever?

## B1. O que torna uma granularidade boa

Agregar sempre reduz zeros, então "menos zeros" sozinho levaria direto ao trimestral. A escolha equilibra ganhos e perdas:

| Critério | Pergunta | Onde é medido |
|---|---|---|
| Poucos zeros | O modelo terá algo além de zero para aprender? | B3 |
| Diferença entre produtos | O grão distingue produtos de venda alta e baixa? | B3 |
| Padrão no passado | A venda de um período ajuda a prever o próximo? | B3 |
| Histórico | Quantos pontos cada série tem para treinar? | B3 |
| Horizonte | Quantos passos até cobrir os 3 meses do plano? | B3 |
| Sinal da estação | Quanto da variação a estação explica? | B4 e B5 |
| Ruído do alvo | Quanto do valor previsto é variação aleatória? | B6 |

## B2. Períodos incompletos

**Como medimos:** um período é incompleto quando tem menos dias que o calendário (`days_in_period` menor que 7 na semana, que os dias do mês ou do trimestre). A tabela compara a taxa de zeros e a demanda média de períodos completos e incompletos.

**Por que importa:** períodos incompletos distorcem a série e precisam ficar fora das comparações e do treino.

In [ ]:
partial = pd.concat(
    [
        add_partial_period_flag(df, GRAINS[name].date_col, GRAINS[name].grain)
        .groupBy("is_partial_period")
        .agg(
            F.count("*").alias("n_rows"),
            F.round(F.mean((F.col("demand_quantity") == 0).cast("int")), 4).alias("zero_rate"),
            F.round(F.mean(F.col("demand_quantity").cast("double")), 3).alias("avg_demand"),
        )
        .toPandas()
        .assign(grao=name)
        for name, df in DFS.items()
        if name != "diario"
    ]
)
partial["pct_rows"] = (partial["n_rows"] / partial.groupby("grao")["n_rows"].transform("sum")).round(4)

partial.set_index(["grao", "is_partial_period"])

**Resultado** *(16/09/2026)*

| Grão | Linhas incompletas | Zeros: completos × incompletos |
|---|---|---|
| Semanal | 0,6% | 94,5% × 52,3% |
| Mensal | 2,6% | 78,6% × 44,4% |
| Trimestral | 7,7% | 49,9% × 31,7% |

**O que significa:** os períodos incompletos têm **menos** zeros, não mais. O primeiro período de cada série sempre tem venda, porque a série começa justamente nela. Eles distorcem a série e ficam fora de todas as comparações e do treino. O trimestral é o grão mais afetado.

## B3. Os quatro grãos lado a lado

**Como medimos:** um quadro com os critérios da B1 para cada grão, só com períodos completos. É a célula mais pesada do notebook (alguns minutos).

| Linha do quadro | O que mede |
|---|---|
| % zeros | Fração dos períodos sem venda |
| % zeros — 10% que menos vendem / que mais vendem | A diferença entre os dois mostra se o grão distingue os produtos |
| intervalo típico entre vendas | Mediana de períodos entre uma venda e outra |
| % séries regulares / irregulares | Classes smooth e lumpy da A5 |
| % séries com padrão no período anterior | Séries em que a venda de um período ajuda a prever o seguinte, além do acaso (por puro acaso, seriam ~2,5%) |
| % séries com padrão anual | O mesmo, comparando com o mesmo período do ano anterior |
| ruído de venda | Variância ÷ média da série. Perto de 1 = vendas ao acaso; acima de 1 = pedidos com vários itens ou vendas agrupadas |
| pontos por série / passos até 3 meses | Histórico disponível e passos até o horizonte |

In [ ]:
FRIENDLY = {
    "zero_rate": "% zeros",
    "zero_rate_decil_1": "% zeros — 10% que menos vendem",
    "zero_rate_decil_10": "% zeros — 10% que mais vendem",
    "adi_mediano": "intervalo típico entre vendas (períodos)",
    "pct_smooth": "% séries regulares",
    "pct_lumpy": "% séries irregulares",
    "pct_series_acf_lag1_signif": "% séries com padrão no período anterior",
    "pct_series_acf_sazonal_signif": "% séries com padrão anual",
    "dispersao_mediana": "ruído de venda (variância ÷ média)",
    "periodos_por_serie": "pontos por série",
    "passos_ate_3_meses": "passos até 3 meses",
    "pct_periodos_parciais": "% períodos incompletos",
}

scorecard = pd.DataFrame([grain_scorecard(df, name, GRAINS[name]) for name, df in DFS.items()]).set_index("grao")

table = scorecard[list(FRIENDLY)].copy()
pct_cols = [c for c in table.columns if c.startswith(("pct_", "zero_rate"))]
table[pct_cols] = (table[pct_cols].astype(float) * 100).round(1)

# astype(object) antes de transpor: mantém inteiros como inteiros na tabela.
table.rename(columns=FRIENDLY).astype(object).T

**Resultado** *(16/09/2026)*

| Critério | Diário | Semanal | Mensal | Trimestral |
|---|---|---|---|---|
| % zeros | 99,2 | 94,5 | 78,6 | 49,9 |
| % zeros: 10% que menos → que mais vendem | 99,4 → 98,9 | 96,3 → 92,6 | 85,3 → 72,3 | 62,7 → 39,1 |
| Intervalo típico entre vendas | 118 dias | 18 semanas | 4,7 meses | 2 trimestres |
| % séries com padrão no período anterior | 6,3 | 6,6 | 5,8 | 1,2 |
| % séries com padrão anual | 6,4 | 7,9 | 7,9 | 7,6 |
| Ruído de venda (variância ÷ média) | 7,0 | 6,9 | 6,9 | 7,1 |
| Pontos por série | 2.284 | 327 | 76 | 26 |
| Passos até 3 meses | 91 | 13 | 3 | 1 |
| % períodos incompletos | 0 | 0,6 | 2,6 | 7,7 |

**O que significa, critério a critério:**

- **Zeros e diferença entre produtos:** diário e semanal têm mais de 90% de zeros e quase não distinguem produtos de venda alta e baixa. Ficam descartados.
- **Padrão no passado:** em todos os grãos, só 6–8% das séries têm padrão além do acaso (~2,5%). **O histórico de um produto numa loja quase não ajuda a prever o próprio futuro.** No trimestral cai para 1,2% porque, com ~26 pontos por série, não há dado suficiente para detectar padrão nenhum.
- **Ruído de venda:** fica em ~7 em todos os grãos, com leve alta no trimestral (6,9 → 7,1). O número mede o quanto a venda de uma série oscila em torno da própria média, e aqui ele reflete principalmente o **tamanho do pedido**: quando vende, vende ~5,5 unidades de uma vez.

  **Isso não contradiz a sazonalidade.** A estação muda a *média* de cada mês (numa série de 1,4 un./mês, ~2,4 em janeiro e ~0,6 em julho), mas o sobe e desce do acaso é bem maior (±3,1 un. num mês). Dentro de uma série, a estação responde por só 3,8% da variação (B6); o resto é acaso. Por isso ela quase não muda esta medida — e só aparece com clareza quando se somam séries, onde explica 60,5%.
- **Histórico e horizonte:** o trimestral tem só 26 pontos por série e só permite prever a partir de janeiro, abril, julho e outubro. O mensal tem 76 pontos e cobre os 3 meses em 3 passos, a partir de qualquer mês.
- **Períodos incompletos:** o trimestral tem 3× mais que o mensal.

**Resumo:** diário e semanal saem. O trimestral ganha em zeros, mas perde em histórico, flexibilidade e bordas. O mensal é o melhor equilíbrio; a B5 e a B6 confirmam.

**Em palavras simples:** pense nos quatro grãos como quatro formas de "juntar" as vendas antes de olhar para elas. Diário e semanal juntam pouco: sobra quase tudo zero, então é como tentar enxergar um padrão numa folha quase em branco. Trimestral junta demais: reduz os zeros, mas cada série passa a ter só ~26 pontos de histórico (pouco mais de 6 anos de dados vira 26 números), o que é pouco para o modelo aprender, e só dá para começar a prever em 4 meses do ano. **Mensal é o equilíbrio**: ainda tem bastante zero (79%), mas é normal para esse tipo de produto (seção A), tem histórico suficiente (76 pontos por série) e permite prever a partir de qualquer mês, cobrindo os 3 meses seguintes em 3 passos. A linha "padrão no período anterior" e "padrão anual", perto de 6–8% em todos os grãos, é outra forma de dizer o que a seção 8 do notebook 03 confirma depois: **olhar só para o passado de um produto quase não ajuda a prever o futuro dele**, não importa se você olha dia, semana, mês ou trimestre.

## B4. A demanda tem estação?

**Como medimos:** `int_high_season_months` soma a demanda de cada mês, calcula a média de cada mês do ano ao longo dos anos e divide pela média dos 12 meses (`seasonal_index`; 1,0 = mês típico). Um mês é **alta temporada** quando fica acima da média. É a mesma tabela que alimenta `dim_date.is_high_season`.

In [ ]:
spark.table(daily_config.silver_table("int_high_season_months")).orderBy("month_number").toPandas()

**Resultado** *(16/09/2026)*

| Mês | jan | fev | mar | abr | mai | jun | jul | ago | set | out | nov | dez |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Índice | 1,77 | 1,39 | 1,36 | 1,05 | 0,71 | 0,47 | 0,42 | 0,51 | 0,64 | 0,99 | 1,22 | 1,46 |

**O que significa:** a estação é forte. Janeiro vende 4× mais que julho. A alta temporada vai de **novembro a abril**. Outubro fica logo abaixo da média (0,99) e pode entrar conforme o histórico crescer, porque a regra é recalculada com os dados.

Para o modelo, a marcação sozinha é grosseira: trata abril (1,05) igual a janeiro (1,77). Por isso o **índice sazonal** dos meses previstos também entra como informação.

## B5. Quanto da variação a estação explica?

**Como medimos:** para cada série, prevê cada período como "média da série × índice do mês" e mede quanto da variação essa conta explica (R² sazonal: 0% = a estação não aparece; 100% = a série é só estação). Feito em dois níveis:

- **produto × loja:** a série que vamos prever;
- **categoria × loja:** os produtos da mesma categoria somados — mostra quanto sinal existe quando se junta produtos parecidos.

O índice é calculado nos mesmos dados em que é avaliado, então o R² é um teto otimista — serve para comparar grãos e níveis entre si. Célula pesada: reagrega a série diária oito vezes.

In [ ]:
segmented_all = add_segment_columns(daily_df, daily_config.gold_table, date_col=None)
LEVELS = {
    "produto_x_loja": ["sk_product", "sk_location"],
    "categoria_x_loja": ["category_name", "sk_location"],
}

signal_rows = []
for level, ids in LEVELS.items():
    for name, config in GRAINS.items():
        series = aggregate_series(segmented_all, ids, config.grain).filter(~F.col("is_partial_period"))
        row = seasonal_signal_share(series, "period_start_date", ids).collect()[0].asDict()
        row["zero_rate"] = round(series.agg(F.mean((F.col("demand_quantity") == 0).cast("int"))).collect()[0][0], 4)
        signal_rows.append({"nivel": level, "grao": name, **row})

signal = pd.DataFrame(signal_rows)
signal.pivot(index="grao", columns="nivel", values=["zero_rate", "r2_seasonal"]).loc[list(GRAINS)]

**Resultado** *(16/09/2026)*

| Grão | produto × loja: zeros | produto × loja: estação explica | categoria × loja: zeros | categoria × loja: estação explica |
|---|---|---|---|---|
| Diário | 99,2% | 0,1% | 62,0% | 12,2% |
| Semanal | 94,5% | 0,9% | 21,1% | 41,2% |
| Mensal | 78,6% | 3,8% | 13,6% | 60,5% |
| Trimestral | 49,9% | 9,3% | 8,1% | 63,6% |

**O que significa:**

- **Numa série produto × loja, a estação explica só 3,8% da variação mensal.** O resto é ruído de venda. A estação existe (B4), mas fica escondida dentro de cada série.
- **Somando os produtos da mesma categoria na loja, a estação explica 60,5%.** O sinal aparece quando se junta produtos parecidos.
- **Do mensal para o trimestral o ganho é pequeno** (60,5% → 63,6%), enquanto do semanal para o mensal é grande (41,2% → 60,5%). O mensal captura quase todo o sinal.

**Consequência para o modelo:** a previsão continua sendo por **produto × loja**, mas o modelo deve receber informação de produtos parecidos, como a venda da categoria na mesma loja e a venda do produto nas outras lojas. É assim que ele aproveita o sinal que a série sozinha não mostra.

**Em palavras simples:** a coluna "estação explica" responde à pergunta "se eu só soubesse em que mês estamos, quanto disso eu já teria adivinhado da venda?" — 0% seria "nada, é tudo imprevisível", 100% seria "tudo, a série é só a estação, sem surpresas". Para um produto sozinho numa loja, a resposta é quase nada (3,8%): a venda de um item específico é dominada por sorte (comprou ou não comprou um cliente naquele mês), e a estação, mesmo sendo real, fica escondida atrás desse ruído. Mas ao somar todos os produtos de uma categoria numa loja, o ruído de cada produto se cancela e a estação aparece claramente (60,5%). **Por isso o modelo precisa "ver" a venda do grupo, não só a do produto isolado** — é a mesma lição do notebook 03 (seção 7).

## B6. O que prever: um mês ou o total de 3 meses?

**Como medimos:** na série mensal (meses completos), compara dois alvos possíveis:

- **um mês:** a demanda de cada mês;
- **total de 3 meses:** a soma dos 3 meses seguintes a cada mês.

Para cada alvo: a taxa de zeros e o **ruído relativo** (`noise_ratio`), o tamanho aproximado da variação aleatória em relação à média. É o erro que sobra mesmo com um modelo perfeito: 2,0 = a variação aleatória é o dobro da média.

In [ ]:
monthly_config = GRAINS["mensal"]
monthly_complete = add_partial_period_flag(DFS["mensal"], monthly_config.date_col, "month").filter(
    ~F.col("is_partial_period")
)

horizon_target_comparison(monthly_complete, monthly_config.date_col, horizon=3).toPandas()

**Resultado** *(16/09/2026)*

| Alvo | Alvos sem venda | Média do alvo | Ruído relativo |
|---|---|---|---|
| Um mês | 78,6% | 1,4 un. | 2,25 |
| **Total de 3 meses** | **50,0%** | **4,1 un.** | **1,31** |

**O que significa:** prever o total dos 3 meses seguintes corta os alvos sem venda de 79% para 50% e reduz o ruído relativo quase pela metade (2,25 → 1,31). Mesmo um modelo perfeito erraria o total de 3 meses de uma série em algo do tamanho da própria média — esse é o piso de erro por série. O ganho do modelo vai aparecer na soma de muitas séries (por loja, por categoria, no total da compra).

**Em palavras simples:** "ruído relativo" é uma estimativa de quanto erro sobra mesmo com uma previsão perfeita, só por causa do acaso natural da venda. Um ruído de 2,25 significa que a variação aleatória é mais que o dobro da própria média prevista — ou seja, mesmo acertando "na média", o valor real de um mês específico pode variar bastante para cima ou para baixo, só por sorte. Ao somar 3 meses em vez de prever mês a mês, esse ruído cai para 1,31: como somar várias jogadas de dados em vez de julgar por uma única jogada, o resultado fica mais estável. Por isso escolhemos prever o total de 3 meses: não elimina o acaso, mas reduz bastante o piso de erro que nenhum modelo consegue superar.

## Conclusão

**Em uma frase, para quem não trabalha com dados:** a venda de cada produto numa loja é naturalmente rala e cheia de "zeros" — isso não é erro de dado, é assim que o negócio vende — e a forma que melhor equilibra ter poucos zeros com ter histórico suficiente para aprender é olhar mês a mês e prever o total dos 3 meses seguintes, em vez de olhar dia a dia ou prever só o próximo mês.

| Decisão | Por quê | Seção |
|---|---|---|
| **Manter os zeros** | São venda baixa real, não erro de dado | Parte A |
| **Série mensal por produto × loja** | Diário e semanal têm zeros demais; o trimestral tem pouco histórico e só prevê a partir de inícios de trimestre | B3 |
| **Alvo = total dos 3 meses seguintes** | 50% de zeros (contra 79% no mês) e metade do ruído | B6 |
| **Remover períodos incompletos** | Distorcem a série: o primeiro sempre tem venda | B2 |
| **Usar o sinal de produtos parecidos como feature** | A estação explica 3,8% de uma série, mas 60,5% quando os produtos da categoria na loja são somados | B5 |
| **Usar o índice sazonal, não só a alta temporada** | A marcação trata abril (1,05) igual a janeiro (1,77) | B4 |
| **Não confiar só no histórico da própria série** | Só 6–8% das séries têm padrão além do acaso | B3 |

**Próximo passo:** o notebook `02_distribuicao_produtos` divide as séries em grupos e define o que cada grupo recebe nas etapas seguintes.